# Exploratory Data Analysis — Credit Card Fraud Detection

**Dataset**: [ULB Credit Card Fraud](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
**Features**: 30 columns — `Time`, `V1–V28` (PCA-anonymised), `Amount`  
**Target**: `Class` — 0 = legitimate, 1 = fraud

Key challenges explored here:
- Severe class imbalance (~0.17% fraud)
- No raw feature names (PCA-transformed for privacy)
- Temporal ordering: transactions arrive as a stream

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

df = pd.read_csv('../data/creditcard.csv')
print(f'Shape: {df.shape}')
df.head()

## 1. Class imbalance

The single most important property of this dataset. Only ~0.17% of transactions are fraudulent.
Standard accuracy is meaningless here — a classifier predicting "legit" for everything scores 99.83%.

In [ ]:
counts = df['Class'].value_counts()
fraud_rate = df['Class'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Legitimate', 'Fraud'], counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Transaction count by class')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(
    counts.values,
    labels=['Legitimate', 'Fraud'],
    colors=['#2ecc71', '#e74c3c'],
    autopct='%1.3f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title(f'Fraud rate: {fraud_rate:.3f}%')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Legitimate: {counts[0]:,} | Fraud: {counts[1]:,} | Ratio: {counts[0]/counts[1]:.0f}:1')

## 2. Transaction amount — fraud vs. legitimate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

legit = df[df['Class'] == 0]['Amount']
fraud = df[df['Class'] == 1]['Amount']

axes[0].hist(legit, bins=100, alpha=0.6, color='#2ecc71', label='Legitimate', density=True)
axes[0].hist(fraud, bins=100, alpha=0.6, color='#e74c3c', label='Fraud', density=True)
axes[0].set_xlabel('Amount (€)')
axes[0].set_ylabel('Density')
axes[0].set_title('Amount distribution (raw)')
axes[0].set_xlim(0, 2000)
axes[0].legend()

axes[1].boxplot([legit, fraud], labels=['Legitimate', 'Fraud'], patch_artist=True,
                boxprops={'facecolor': '#ecf0f1'})
axes[1].set_ylabel('Amount (€)')
axes[1].set_title('Amount box plot (log scale)')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print(f"Fraud   — median: €{fraud.median():.2f}  |  mean: €{fraud.mean():.2f}  |  max: €{fraud.max():.2f}")
print(f"Legit   — median: €{legit.median():.2f}  |  mean: €{legit.mean():.2f}  |  max: €{legit.max():.2f}")

## 3. Fraud rate over time

`Time` is seconds elapsed since the first transaction. Looking at rolling fraud rate reveals
whether fraud clusters at specific times of day (a useful signal).

In [ ]:
df_sorted = df.sort_values('Time').copy()
df_sorted['hour'] = (df_sorted['Time'] // 3600) % 24

hourly = df_sorted.groupby('hour')['Class'].agg(['sum', 'count'])
hourly['fraud_rate'] = hourly['sum'] / hourly['count'] * 100

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(hourly.index, hourly['fraud_rate'], color='#e74c3c', alpha=0.75)
ax.set_xlabel('Hour of day')
ax.set_ylabel('Fraud rate (%)')
ax.set_title('Fraud rate by hour of day')
ax.axhline(df['Class'].mean() * 100, color='black', linestyle='--', label='Overall rate')
ax.legend()
plt.tight_layout()
plt.show()

## 4. PCA feature distributions (V1–V28)

Even though the features are anonymised PCA components, the *separation* between
fraud and legitimate distributions tells us which ones carry the most signal.

In [ ]:
v_cols = [f'V{i}' for i in range(1, 29)]

# Compute mean absolute difference between fraud and legit for each feature
fraud_means = df[df['Class'] == 1][v_cols].mean()
legit_means = df[df['Class'] == 0][v_cols].mean()
separation = (fraud_means - legit_means).abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#e74c3c' if separation[f] > separation.median() else '#95a5a6' for f in separation.index]
ax.bar(separation.index, separation.values, color=colors)
ax.set_xlabel('Feature')
ax.set_ylabel('|mean_fraud − mean_legit|')
ax.set_title('Mean separation between fraud and legitimate (top features highlighted)')
ax.axhline(separation.median(), color='black', linestyle='--', alpha=0.5, label='Median')
ax.legend()
plt.tight_layout()
plt.show()

print('Top 5 most separating features:', list(separation.head().index))

## 5. Correlation heatmap of top features

In [ ]:
top_features = list(separation.head(12).index) + ['Amount', 'Class']
corr = df[top_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, linewidths=0.5, ax=ax, annot_kws={'size': 9}
)
ax.set_title('Correlation heatmap — top discriminating features + Amount')
plt.tight_layout()
plt.show()

## Key findings

| Finding | Implication for modelling |
|---|---|
| 0.17% fraud rate (492 / 284,807) | Use `class_weight='balanced'` or SMOTE; track PR-AUC not accuracy |
| Fraud amounts skew lower than legitimate | `Amount` alone is a weak feature; model needs V-features |
| V4, V11, V12, V14 show strong separation | LightGBM will likely rank these highly in feature importance |
| No strong inter-feature correlation | PCA pre-processing done upstream; no need for further dimensionality reduction |